# How much load can one bus take?

The map-seed sweep left one question open. `Reordering W900` — the best
policy on every map tested — still misses the design review's **80% fulfilment
gate on 4 of 25 maps**, bottoming out at 65.5%. This asks whether that is a
routing failure or a **load** failure.

The distinction matters for what you would write down. If coverage recovers as
demand thins, then those maps are simply asking one vehicle to do more than one
vehicle can, and the honest statement is about fleet size rather than about
the algorithm. If it does not recover, the routing has a real limitation.

**There is only one bus.** Nothing in the simulation models a fleet, so the
arrival rate a single vehicle can absorb is the whole of its capacity.

## What it varies

- **Demand rate** — the interval between requests, from one every 30 s to one
  every 300 s. This is the load axis.
- **Capacity** — a secondary check. The reordering headroom study found
  rejections are caused by the *deadline* rather than by seats, so capacity is
  expected to do very little; this is where that gets tested rather than
  assumed.
- **Maps** — five, chosen by measured difficulty rather than by taste, so the
  curve is read across a range rather than on one instance.

Stopping is safe at any point; re-running continues. Prefer a terminal over
Jupyter for the run — workers are forked.

## Configuration

In [ ]:
from src.experiment.store import DEFAULT_RUNS_DIR

# Chosen from the 25-map sweep by served fraction under Reordering W900:
# hardest, 25th percentile, median, 75th, easiest. Map 17 is the one that
# bottoms out at 65.5%; map 15 reaches 98.8%.
MAPS = {17: "hardest", 24: "25th pct", 10: "median", 42: "campus (75th)",
        15: "easiest"}

# stop_density=0.5 gives the 21-stop campus map. The default 0.15 gives 6 and
# makes every comparison collapse -- see route_reordering.md.
MAP_SHAPE = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0),
                 stop_density=0.5)

# The load axis. 60 s is the standard used everywhere else, so the curve
# passes through every existing result.
INTERVALS = [30.0, 45.0, 60.0, 90.0, 120.0, 180.0, 300.0]

# Secondary axis, swept only at the standard interval.
CAPACITIES = [10, 20, 40, 80]

BASE_TRIAL = dict(max_requests=200, capacity=40, max_time=200000.0)
PATIENCE = 900.0          # passenger patience; the window is set to match
WINDOW = 900.0
SEEDS = list(range(30))

RUNS = DEFAULT_RUNS_DIR
N_JOBS = 10

In [ ]:
import math
import statistics
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from scipy import stats

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import run_campaign
from src.experiment.store import load_runs
from src.map.graph import PlanarStreetGraph

from IPython.display import Markdown, display

# Validated palette slots 1-3 (all-pairs CVD 9.2, normal-vision 24.0 on this
# surface). Three is the cap that clears all-pairs; a fourth series folds into
# "Other" or gets a facet of its own.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
PRIMARY = SERIES[0]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})


def mean_of(runs, field):
    """
    Average one KPI over a set of runs.
    :param runs: RunRecords.
    :param field: KPISnapshot attribute name.
    :return: The mean, or NaN when nothing is defined.
    """
    values = [getattr(r.kpi, field) for r in runs
              if getattr(r.kpi, field) is not None]
    return statistics.fmean(values) if values else float("nan")


def half_width(values, confidence=0.95):
    """
    95% confidence half-width for a sample.
    :param values: The sample.
    :param confidence: Confidence level.
    :return: The half-width, or 0.0 when it is undefined.
    """
    if len(values) < 2:
        return 0.0
    return (stats.t.ppf(1 - (1 - confidence) / 2, len(values) - 1)
            * statistics.stdev(values) / len(values) ** 0.5)


POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Window W900", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering W900",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

total = (len(MAPS) * len(POLICIES) * len(SEEDS)
         * (len(INTERVALS) + len(CAPACITIES)))
print(f"{len(MAPS)} maps x {len(POLICIES)} policies x {len(SEEDS)} seeds "
      f"x ({len(INTERVALS)} intervals + {len(CAPACITIES)} capacities) "
      f"= {total:,} trials")
print("max_time is raised to 200,000 s: at one request every 300 s the "
      "demand alone spans 60,000 s, and a ceiling that cuts a run short "
      "would look exactly like a coverage failure.")

## Run it

Two sweeps. The demand-rate sweep holds capacity at 40; the capacity sweep
holds the interval at 60 s. A full cross product would be four times the work
to answer a question the headroom study says is already settled — if capacity
turns out to matter here, that is the surprise worth following up.

In [ ]:
def world_for(map_seed):
    """
    Describe the map, including everything that changes its geometry.
    :param map_seed: Seed handed to PlanarStreetGraph.generate.
    :return: The world identity.
    """
    return {
        "map": "planar", "n_seeds": MAP_SHAPE["n_seeds"],
        "bounds": list(MAP_SHAPE["bounds"]),
        "stop_density": MAP_SHAPE["stop_density"],
        "map_seed": map_seed, "backend": "MockBackend",
    }


def sweep(axis_name, values, **fixed):
    """
    Run one campaign per (map, value) and report progress.
    :param axis_name: Trial parameter being swept.
    :param values: Values to sweep it over.
    :param fixed: Trial parameters held constant.
    :return: None; everything lands in the store.
    """
    started = time.perf_counter()
    ran = skipped = 0
    for map_seed in MAPS:
        graph = PlanarStreetGraph.generate(**MAP_SHAPE, seed=map_seed)
        for value in values:
            trial = {**BASE_TRIAL, **fixed, axis_name: value}
            result = run_campaign(
                policies=POLICIES, seeds=SEEDS, world=world_for(map_seed),
                directory=RUNS, graph=graph, n_jobs=N_JOBS, progress=False,
                max_wait_s=PATIENCE, **trial)
            ran += result.ran
            skipped += result.skipped
        print(f"  map {map_seed:<3} ran={ran:<6} skipped={skipped:<6} "
              f"{(time.perf_counter() - started) / 60:5.1f} min")


print("demand-rate sweep")
sweep("demand_interval_s", INTERVALS)
print("capacity sweep")
sweep("capacity", CAPACITIES, demand_interval_s=60.0)
print("done")

## Collect

In [ ]:
LABELS = [label for label, _ in POLICIES]
records = [
    r for r in load_runs(RUNS)
    if r.identity.world.get("stop_density") == 0.5
    and r.identity.world.get("map_seed") in MAPS
    and r.identity.trial.get("max_wait_s") == PATIENCE
    and r.identity.trial.get("max_requests") == BASE_TRIAL["max_requests"]
    and r.identity.policy in LABELS
]
rows = []
for r in records:
    rows.append({
        "map": r.identity.world["map_seed"],
        "difficulty": MAPS[r.identity.world["map_seed"]],
        "policy": r.identity.policy,
        "interval": r.identity.trial["demand_interval_s"],
        "capacity": r.identity.trial["capacity"],
        "seed": r.identity.seed,
        "served": r.kpi.served_fraction * 100,
        "wait": r.kpi.mean_wait_time,
        "ride": r.kpi.mean_ride_time,
    })
frame = pd.DataFrame(rows)
print(f"{len(frame)} runs")
print(frame.groupby(["interval", "capacity"]).size().to_string())

## 1. The saturation curve

Served fraction against demand rate, one panel per map, ordered hardest to
easiest. The dashed line is the design review's 80% gate.

**What to look for:** whether every map eventually clears 80% as demand thins.
If it does, the gate is a statement about load rather than about routing.

In [ ]:
demand = frame[frame["capacity"] == BASE_TRIAL["capacity"]]
ordered_maps = list(MAPS)

# squeeze=False so a single-map configuration still yields an array.
figure, axes = plt.subplots(1, len(ordered_maps),
                            figsize=(3.0 * len(ordered_maps), 3.4),
                            sharey=True, squeeze=False)
axes = axes[0]
for axis, map_seed in zip(axes, ordered_maps):
    subset = demand[demand["map"] == map_seed]
    for colour, policy in zip(SERIES, LABELS):
        grouped = subset[subset["policy"] == policy].groupby("interval")
        means = grouped["served"].mean()
        errs = grouped["served"].apply(lambda v: half_width(list(v)))
        axis.errorbar(means.index, means.values, yerr=errs.values,
                      marker="o", markersize=4, lw=1.8, color=colour,
                      capsize=2, label=policy)
    axis.axhline(80, ls="--", lw=1.1, color=INK_SOFT)
    axis.set_title(f"map {map_seed} — {MAPS[map_seed]}")
    axis.set_xlabel("seconds between requests")
    axis.grid(axis="y")
    axis.set_axisbelow(True)
axes[0].set_ylabel("served (%)")
axes[-1].legend(loc="lower right")
figure.tight_layout()
plt.show()

In [ ]:
print("Slowest demand rate at which each map still misses 80% "
      "(Reordering W900):")
best = demand[demand["policy"] == "Reordering W900"]
for map_seed in ordered_maps:
    means = best[best["map"] == map_seed].groupby("interval")["served"].mean()
    failing = [i for i, v in means.items() if v < 80]
    reached = [i for i, v in means.items() if v >= 80]
    verdict = (f"never below 80%" if not failing else
               (f"clears 80% from {min(reached):.0f} s onward"
                if reached else "**never reaches 80% at any rate tested**"))
    print(f"  map {map_seed:<3} {MAPS[map_seed]:<15} {verdict}")

## 2. Does capacity matter?

The reordering headroom study concluded that insertion rejections are caused
by the **deadline** rather than by seats — the no-window control rejected
nothing at all. If that holds, quadrupling capacity should barely move
coverage, and the bus's limit is time rather than space.

In [ ]:
seats = frame[frame["interval"] == 60.0]
figure, axis = plt.subplots(figsize=(7.5, 4.0))
for colour, policy in zip(SERIES, LABELS):
    grouped = seats[seats["policy"] == policy].groupby("capacity")
    means = grouped["served"].mean()
    errs = grouped["served"].apply(lambda v: half_width(list(v)))
    axis.errorbar(means.index, means.values, yerr=errs.values, marker="o",
                  markersize=5, lw=1.8, color=colour, capsize=2, label=policy)
axis.axhline(80, ls="--", lw=1.1, color=INK_SOFT)
axis.set_xlabel("capacity (seats)")
axis.set_ylabel("served (%), pooled over the five maps")
axis.set_title("Seats bought against coverage gained")
axis.set_xscale("log", base=2)
axis.set_xticks(CAPACITIES, [str(c) for c in CAPACITIES])
axis.grid(axis="y")
axis.set_axisbelow(True)
axis.legend()
figure.tight_layout()
plt.show()

pooled = seats.groupby(["policy", "capacity"])["served"].mean().unstack()
display(pooled.round(1))
span = pooled.max(axis=1) - pooled.min(axis=1)
display(Markdown(
    "Coverage moved by **"
    + ", ".join(f"{p}: {v:.1f} pp" for p, v in span.items())
    + f"** across an {min(CAPACITIES)}→{max(CAPACITIES)} seat range."))

## 3. What to conclude

Fill in once run. The claim this notebook exists to settle:

> The 80% gate is missed on hard maps because **one bus cannot absorb one
> request every 60 s on a 2 km map**, not because the routing is inadequate.

It is supported if coverage on map 17 climbs through 80% as the interval
lengthens, and if capacity moves it very little. It is refuted if map 17 stays
below 80% even at one request every 300 s — which would mean something is
wrong beyond load, and would be worth chasing.